In [ ]:
# ==================== 2. EVALUATE ECAPA-TDNN ====================
import sys
sys.path.append(os.path.abspath("./ECAPA"))
from predict_emotion import EmotionClassifier
from train_emotion_model import prepare_features, AudioFeaturesDataset, collate_fn, evaluate
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ecapa_model = EmotionClassifier(num_labels).to(device)

checkpoint_path = "./ECAPA/emotion_model/best_ecapa_model.pth"
if not os.path.exists(checkpoint_path):
    print(f"❌ ECAPA checkpoint not found at {checkpoint_path}. Please retrain.")
else:
    print(f"✨ Loading ECAPA-TDNN model from {checkpoint_path}...")
    ecapa_model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    ecapa_model.eval()
    
    print("Extracting Mel-spectrogram features for Test set (this takes a moment)...")
    X_test_feat, y_test_clean = prepare_features(X_test, y_test, "Test")
    test_dataset = AudioFeaturesDataset(X_test_feat, y_test_clean)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
    
    print("Running Inference...")
    test_preds, test_labels_out = evaluate(ecapa_model, test_loader, device)
    
    ecapa_f1_weighted = f1_score(test_labels_out, test_preds, average='weighted')
    ecapa_acc = accuracy_score(test_labels_out, test_preds)
    
    print(f"\n

In [ ]:
# ==================== 3. EVALUATE DFAT HYBRID FUSION ====================
sys.path.append(os.path.abspath("./DFAT Hybrid Fusion"))
import pickle
from transformers import AutoFeatureExtractor, AutoModel, WhisperProcessor, WhisperForConditionalGeneration, AutoTokenizer

model_dir = Path("./DFAT Hybrid Fusion/dualstream_model")
if not (model_dir / "metadata.json").exists():
    print("❌ DFAT models not found. Please run train_dualstream.py first.")
else:
    print("✨ Loading Ensemble Models and Weights...")
    with open(model_dir / "lr_model.pkl", "rb") as f: lr_model = pickle.load(f)
    with open(model_dir / "rf_model.pkl", "rb") as f: rf_model = pickle.load(f)
    with open(model_dir / "xgb_model.pkl", "rb") as f: xgb_model = pickle.load(f)
    with open(model_dir / "scaler.pkl", "rb") as f: scaler = pickle.load(f)
    with open(model_dir / "metadata.json", "r") as f: metadata = json.load(f)
    
    w_xgb, w_rf, w_lr = metadata["ensemble_weights"]["xgb"], metadata["ensemble_weights"]["rf"], metadata["ensemble_weights"]["lr"]
    print(f"Ensemble Weights -> XGB: {w_xgb:.3f}, RF: {w_rf:.3f}, LR: {w_lr:.3f}")
    
    print("\nLoading Extractors (WavLM, Whisper, PhoBERT)...")
    wavlm_processor = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
    wavlm_model = AutoModel.from_pretrained("microsoft/wavlm-base-plus").to(device).eval()
    
    whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device).eval()
    
    phobert_tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
    phobert_model = AutoModel.from_pretrained("vinai/phobert-base-v2").to(device).eval()
    
    from train_dualstream import extract_features_for_split
    
    print("\nExtracting Dual-Stream Features for Test Set (this will take several minutes)...")
    # This invokes audio loading, Whisper ASR, Underthesea word segmentation, and PhoBERT embedding.
    # It automatically uses transcript_cache.json if available.
    test_fused, test_labels_ext = extract_features_for_split(
        X_test, y_test, wavlm_model, wavlm_processor, whisper_model, whisper_processor, 
        phobert_model, phobert_tokenizer, device, "Test"
    )
    
    # Scale features
    test_fused_scaled = scaler.transform(test_fused)
    
    print("\nRunning Late Fusion Ensemble Inference...")
    lr_proba = lr_model.predict_proba(test_fused_scaled)
    rf_proba = rf_model.predict_proba(test_fused_scaled)
    xgb_proba = xgb_model.predict_proba(test_fused_scaled)
    
    ensemble_proba = w_xgb * xgb_proba + w_rf * rf_proba + w_lr * lr_proba
    ensemble_pred = np.argmax(ensemble_proba, axis=1)
    
    dfat_f1_weighted = f1_score(test_labels_ext, ensemble_pred, average="weighted")
    dfat_acc = accuracy_score(test_labels_ext, ensemble_pred)
    
    print(f"\n